In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
import json
import numpy as np
import random

load_dotenv()

# 로컬
# ROOT = Path(os.environ["DATA_ROOT"])
# HF_HOME = ROOT / ".hf_cache"
# os.environ["HF_HOME"] = str(HF_HOME)

# 클라우드
os.environ["HF_HOME"] = ".hf_cache"

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import load_dataset
from sklearn.metrics import f1_score

In [ ]:
# Config
config = {
    'num_labels': 188,
    'seed': 42,
    'learning_rate': 3e-5,
    'batch_size': 8,
    'epochs': 12,
    'early_stop': 5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'model_name': 'skt/A.X-Encoder-base',
    "repo": "ingyoun/A.X-kobert-patent-baseline",
    "run_name": "A.X-Encoder-baseline"
}

In [3]:
random.seed(config['seed'])
np.random.seed(config['seed'])
torch.manual_seed(config['seed'])
torch.cuda.manual_seed_all(config['seed'])

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


## 데이터셋

In [ ]:
dataset = load_dataset("ingyoun/patent-clean-text-kobert-tokenized")
dataset

README.md:   0%|          | 0.00/600 [00:00<?, ?B/s]

c:\workspace\patent_disc\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\workspace\patent_disc\.hf_cache\hub\datasets--ingyoun--patent-clean-text-kobert-tokenized. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00002.parquet:   0%|          | 0.00/543M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/530M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/60.0M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/59.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/201895 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11271 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/11162 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11162
    })
})

## 토크나이저

In [6]:
REV = "38279184ba645e8c94d709fbe92eb5bcb47312c1"
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], trust_remote_code=True, revision=REV)

## 모델

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
        pretrained_model_name_or_path=config["model_name"],
        num_labels=config["num_labels"], 
        problem_type="multi_label_classification", 
        classifier_dropout=0.5
    )

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: int = 2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()

In [ ]:
class FocalTrainer(Trainer):
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self.focal = FocalLoss(0.25, 2.0)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        loss = self.focal(outputs.logits, labels.float())
        return (loss, outputs) if return_outputs else loss

In [ ]:
class MultiLabelCollator:
    def __init__(self, tokenizer):
        self.tok = tokenizer
    
    def __call__(self, feats):
        labels = torch.tensor([f["labels"] for f in feats], dtype=torch.float)
        keys = ("input_ids", "attention_mask")
        enc = [{k: f[k] for k in keys if k in f} for f in feats]
        batch = self.tok.pad(enc, padding=True, return_tensors="pt")
        batch["labels"] = labels
        return batch

In [ ]:
def evaluate_topk(logits, multihot):                        # logits/multihot: [N,188]
    pred_top1 = logits.argmax(axis=1)                       # top-1 예측
    gold_top1 = multihot.argmax(axis=1)                     # 정답 단일화(원본 LabelBinarizer.inverse_transform과 등가)
    out = {
        "weighted_f1": f1_score(gold_top1, pred_top1, average="weighted"),  # baseline headline과 동일
        "micro_f1":    f1_score(gold_top1, pred_top1, average="micro"),
        "macro_f1":    f1_score(gold_top1, pred_top1, average="macro"),
    }
    order = np.argsort(-logits, axis=1)                     # P@k (멀티레이블 참고 지표)
    for k in (1, 3, 5):
        topk = order[:, :k]
        hit = np.take_along_axis(multihot, topk, axis=1).sum(1)
        out[f"p@{k}"] = float((hit / np.clip(multihot.sum(1), 1, k)).mean())
    return out


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return evaluate_topk(np.asarray(logits), np.asarray(labels))

In [ ]:
training_args = TrainingArguments(
    output_dir='/content/results',
    seed=config["seed"],
    learning_rate=config["learning_rate"],
    weight_decay=config["weight_decay"],
    lr_scheduler_type="linear",
    warmup_ratio=config["warmup_ratio"],
    per_device_train_batch_size=config["batch_size"],
    per_device_eval_batch_size=config["batch_size"],
    num_train_epochs=config["epochs"],
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    logging_dir='/content/logs',
    logging_steps=50,
    metric_for_best_model="weighted_f1",
    load_best_model_at_end=True,
    push_to_hub=True,
    hub_model_id=config["repo"],
    hub_strategy="checkpoint",
    report_to="wandb",
    run_name=config["run_name"]
)

In [ ]:
trainer = FocalTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    data_collator=MultiLabelCollator(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stop"])]
)

In [ ]:
trainer.train()

## 평가

In [ ]:
test_metrics = trainer.evaluate(dataset["test"], metric_key_prefix="test")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

In [ ]:
OUT = "/content/out/kobert-baseline"
os.makedirs(OUT, exist_ok=True)
trainer.save_model(OUT)
tokenizer.save_pretrained(OUT)

with open(f"{OUT}/test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)

print("saved", OUT)

In [ ]:
trainer.push_to_hub(config["repo"])
tokenizer.push_to_hub(config["repo"])